# Etape 5 — Tuning manuel CatBoost + MLflow

Objectif : tester au moins 3 configurations d'hyperparametres manuellement,
chacune loggee dans une experience MLflow dediee (`tuning-catboost`).

Modele choisi : **CatBoost** (meilleur modele issu du notebook 11).

## Chargement des donnees et preparation

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

def _find_root(marker="out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError("Racine projet introuvable")

root = _find_root("out")
path = root / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
assert path.exists(), f"CSV introuvable: {path}"

TARGET = "grave"
SEP = ";"

product15_v2 = [
    "dep", "lum", "atm", "catr", "agg", "int", "circ", "col",
    "vma_bucket", "catv_family_4", "manv_mode", "driver_age_bucket",
    "choc_mode", "driver_trajet_family", "time_bucket",
]
cat_cols = product15_v2[:]
MISSING_CAT = "__MISSING__"

df = pd.read_csv(path, sep=SEP)
X = df[product15_v2].copy()
y = df[TARGET].astype(int).copy()

for c in cat_cols:
    X[c] = X[c].astype("string").fillna(MISSING_CAT).astype(str)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape} | positifs : {y_train.mean():.3f}")
print(f"Valid : {X_valid.shape} | positifs : {y_valid.mean():.3f}")

Train : (131620, 15) | positifs : 0.361
Valid : (32906, 15) | positifs : 0.361


## Configuration MLflow

In [2]:
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
)
import matplotlib.pyplot as plt

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "tuning-catboost"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f"MLflow experiment: {MLFLOW_EXPERIMENT}")
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")

MLflow experiment: tuning-catboost
Tracking URI: http://127.0.0.1:5000


## Definition des 4 configurations manuelles

| Config | Strategie |
|--------|----------|
| baseline | Params par defaut raisonnables |
| deep_slow | Arbres profonds + learning rate faible |
| shallow_fast | Arbres peu profonds + learning rate eleve |
| trial1_optuna | Meilleurs params trouves par Optuna (notebook 11) |

In [3]:
# Params incompatibles GPU CatBoost (filtres automatiquement)
GPU_INCOMPATIBLE = {"rsm"}

CONFIGS = {
    "baseline": {
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3.0,
        "iterations": 3000,
    },
    "deep_slow": {
        "depth": 9,
        "learning_rate": 0.02,
        "l2_leaf_reg": 5.0,
        "iterations": 5000,
        "random_strength": 2.0,
        "bagging_temperature": 0.8,
    },
    "shallow_fast": {
        "depth": 4,
        "learning_rate": 0.15,
        "l2_leaf_reg": 1.0,
        "iterations": 2000,
        "bootstrap_type": "MVS",
        "subsample": 0.8,
    },
    "trial1_optuna": {
        "depth": 5,
        "learning_rate": 0.0232,
        "l2_leaf_reg": 0.0555,
        "random_strength": 5.95,
        "bagging_temperature": 1.10,
        "border_count": 114,
        "bootstrap_type": "MVS",
        "subsample": 0.936,
        "iterations": 6000,
    },
}

print(f"{len(CONFIGS)} configurations definies")
for name, params in CONFIGS.items():
    print(f"  - {name}: depth={params['depth']}, lr={params['learning_rate']}")

4 configurations definies
  - baseline: depth=6, lr=0.05
  - deep_slow: depth=9, lr=0.02
  - shallow_fast: depth=4, lr=0.15
  - trial1_optuna: depth=5, lr=0.0232


## Entrainement + logging MLflow pour chaque configuration

In [4]:
from catboost import CatBoostClassifier
import tempfile, os

THRESHOLD = 0.47
EARLY_STOP = 200
TASK_TYPE = "GPU"
results = []

# Metadonnees dataset (etape 6)
n_total = len(y)
n_pos = int(y.sum())
n_neg = n_total - n_pos
dataset_meta = {
    "dataset_size": n_total,
    "train_size": len(y_train),
    "valid_size": len(y_valid),
    "n_features": len(product15_v2),
    "class_0_count": n_neg,
    "class_1_count": n_pos,
    "class_1_ratio": round(n_pos / n_total, 4),
    "class_imbalance_ratio": round(n_neg / n_pos, 4),
}

def _gpu_safe(params):
    """Filtre les params incompatibles GPU et ajoute bootstrap_type si subsample."""
    safe = {k: v for k, v in params.items() if k not in GPU_INCOMPATIBLE}
    if "subsample" in safe and "bootstrap_type" not in safe:
        safe["bootstrap_type"] = "MVS"
    return safe

for config_name, config_params in CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Config: {config_name}")
    print(f"{'='*60}")

    gpu_params = _gpu_safe(config_params)

    with mlflow.start_run(run_name=f"tuning_{config_name}"):
        mlflow.set_tags({
            "config_name": config_name,
            "model_family": "catboost",
            "stage": "tuning_manuel",
        })

        # Log hyperparametres
        for k, v in gpu_params.items():
            mlflow.log_param(k, v)
        mlflow.log_param("threshold", THRESHOLD)
        mlflow.log_param("early_stopping_rounds", EARLY_STOP)
        mlflow.log_param("task_type", TASK_TYPE)

        # Log metadonnees dataset (etape 6)
        mlflow.log_params({f"data_{k}": v for k, v in dataset_meta.items()})

        # Entrainement (GPU accelere — RTX 3060)
        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=0,
            od_type="Iter",
            od_wait=EARLY_STOP,
            task_type=TASK_TYPE,
            **gpu_params,
        )
        model.fit(
            X_train, y_train,
            cat_features=cat_cols,
            eval_set=(X_valid, y_valid),
            use_best_model=True,
        )

        # Predictions sur validation
        proba = model.predict_proba(X_valid)[:, 1]
        preds = (proba >= THRESHOLD).astype(int)

        # Metriques
        metrics = {
            "roc_auc": roc_auc_score(y_valid, proba),
            "pr_auc": average_precision_score(y_valid, proba),
            "f1": f1_score(y_valid, preds),
            "precision": precision_score(y_valid, preds),
            "recall": recall_score(y_valid, preds),
            "accuracy": accuracy_score(y_valid, preds),
            "best_iteration": model.get_best_iteration(),
        }
        mlflow.log_metrics(metrics)

        # Artefacts : matrice de confusion
        with tempfile.TemporaryDirectory() as tmpdir:
            fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
            ConfusionMatrixDisplay.from_predictions(
                y_valid, preds, ax=ax_cm,
                display_labels=["Non grave", "Grave"],
            )
            ax_cm.set_title(f"Confusion Matrix — {config_name}")
            cm_path = os.path.join(tmpdir, "confusion_matrix.png")
            fig_cm.savefig(cm_path, dpi=100, bbox_inches="tight")
            plt.close(fig_cm)
            mlflow.log_artifact(cm_path)

            # Courbe ROC
            fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
            RocCurveDisplay.from_predictions(y_valid, proba, ax=ax_roc)
            ax_roc.set_title(f"ROC Curve — {config_name}")
            roc_path = os.path.join(tmpdir, "roc_curve.png")
            fig_roc.savefig(roc_path, dpi=100, bbox_inches="tight")
            plt.close(fig_roc)
            mlflow.log_artifact(roc_path)

            # Feature importance
            fi = model.get_feature_importance()
            fi_df = pd.DataFrame({
                "feature": product15_v2,
                "importance": fi,
            }).sort_values("importance", ascending=False)
            fi_path = os.path.join(tmpdir, "feature_importance.csv")
            fi_df.to_csv(fi_path, index=False)
            mlflow.log_artifact(fi_path)

            # Feature names (etape 6)
            fn_path = os.path.join(tmpdir, "feature_names.txt")
            with open(fn_path, "w") as f:
                f.write("\n".join(product15_v2))
            mlflow.log_artifact(fn_path)

        # Log du modele
        mlflow.catboost.log_model(model, artifact_path="model")

        # Resume
        print(f"  ROC AUC   : {metrics['roc_auc']:.4f}")
        print(f"  F1        : {metrics['f1']:.4f}")
        print(f"  Precision : {metrics['precision']:.4f}")
        print(f"  Recall    : {metrics['recall']:.4f}")
        print(f"  Best iter : {metrics['best_iteration']}")

        results.append({"config": config_name, **metrics})

print("\nTous les runs sont logges dans MLflow.")


Config: baseline


Default metric period is 5 because AUC is/are not implemented for GPU
2026/02/25 12:22:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ROC AUC   : 0.8212
  F1        : 0.6682
  Precision : 0.6800
  Recall    : 0.6567
  Best iter : 2329
🏃 View run tuning_baseline at: http://127.0.0.1:5000/#/experiments/2/runs/191bac966faa4632a56b8fde630673b2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

Config: deep_slow


Default metric period is 5 because AUC is/are not implemented for GPU
2026/02/25 12:24:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ROC AUC   : 0.8208
  F1        : 0.6680
  Precision : 0.6808
  Recall    : 0.6556
  Best iter : 1698
🏃 View run tuning_deep_slow at: http://127.0.0.1:5000/#/experiments/2/runs/6aaadd91f37540a6a53b618ca166dab1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

Config: shallow_fast


Default metric period is 5 because AUC is/are not implemented for GPU
2026/02/25 12:24:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ROC AUC   : 0.8115
  F1        : 0.6512
  Precision : 0.6813
  Recall    : 0.6236
  Best iter : 1999
🏃 View run tuning_shallow_fast at: http://127.0.0.1:5000/#/experiments/2/runs/28b35bd0b8f448d29685f009729b8361
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

Config: trial1_optuna


Default metric period is 5 because AUC is/are not implemented for GPU
2026/02/25 12:26:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  ROC AUC   : 0.8154
  F1        : 0.6573
  Precision : 0.6823
  Recall    : 0.6342
  Best iter : 5919
🏃 View run tuning_trial1_optuna at: http://127.0.0.1:5000/#/experiments/2/runs/4650f930eb2444aabb854973c03e25a1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

Tous les runs sont logges dans MLflow.


## Comparaison des configurations

In [5]:
results_df = pd.DataFrame(results).set_index("config")
results_df = results_df.sort_values("roc_auc", ascending=False)

print("Classement par ROC AUC :")
display(results_df[["roc_auc", "f1", "precision", "recall", "best_iteration"]])

best_config = results_df.index[0]
print(f"\nMeilleure configuration : {best_config}")
print(f"  ROC AUC = {results_df.loc[best_config, 'roc_auc']:.4f}")
print(f"  F1      = {results_df.loc[best_config, 'f1']:.4f}")
print(f"\nOuvre MLflow UI pour comparer visuellement :")
print(f"  {MLFLOW_TRACKING_URI}/#/experiments")

Classement par ROC AUC :


,roc_auc,f1,precision,recall,best_iteration
config,,,,,
baseline,0.821156,0.668152,0.679983,0.656726,2329
deep_slow,0.820841,0.667983,0.680808,0.655631,1698
trial1_optuna,0.815420,0.657324,0.682255,0.634150,5919
shallow_fast,0.811458,0.651185,0.681299,0.623621,1999



Meilleure configuration : baseline
  ROC AUC = 0.8212
  F1      = 0.6682

Ouvre MLflow UI pour comparer visuellement :
  http://127.0.0.1:5000/#/experiments
